<p align="center">
  <img src="../assets/logo.jpeg" width="100"/>
</p>

# STRATA — ETF Price Direction Prediction

**Binary classification model** to predict whether an ETF's closing price will be higher 3 trading periods ahead, using historical price data and time-series feature engineering.

---

## Pipeline Overview

```
S3 (raw/bronze) → PySpark Feature Engineering → Train/Test Split → Model Training → S3 (silver/gold)
```

**Dataset:** ~3.85M rows · 2,310 tickers · 1993–2021  
**Target:** 1 if `close[t+3] > close[t]`, else 0  
**Validation:** Temporal cutoff split — cutoff date: 2019-01-01  
**Models evaluated:** Logistic Regression, Decision Tree, XGBoost


## 1. Imports

In [ ]:
# Core data and ML libraries
import pandas as pd
import numpy as np
import json
import os
import zipfile
import boto3
import joblib

# Scikit-learn
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report
)

# XGBoost
from xgboost import XGBClassifier

# AWS SageMaker
import sagemaker
from sagemaker import get_execution_role

# PySpark (SageMaker Studio environment)
from pyspark.sql.window import Window
from pyspark.sql.functions import col, lag, avg, std, lead, when, lit, rand

## 2. SageMaker & Spark Session Setup

In [ ]:
# Initialise SageMaker session and retrieve execution role
sagemaker_session = sagemaker.Session()
role = get_execution_role()

# Initialise Spark session (SageMaker Studio built-in)
from sagemaker_studio import sparkutils
spark = sparkutils.init()

# S3 bucket and raw data path
import os

# Load bucket name from environment variable (never hardcode credentials or infra details)
# Set this in your .env file locally — see .env.example at the root of the repo
BUCKET  = "s3://" + os.environ["STRATA_S3_BUCKET"] + "/"
RAW_KEY = "bronze/etf_prices/v1/"

print("SageMaker session ready.")
print(f"Data source: {BUCKET}{RAW_KEY}")

## 3. Data Ingestion

Read the raw ETF price data from S3 (bronze layer, Parquet format).  
Data was originally sourced from Kaggle and ingested into the medallion architecture:
`raw → bronze → silver → gold`


In [ ]:
# Load ETF price data from the bronze S3 layer
df = spark.read.parquet(f"{BUCKET}{RAW_KEY}")

print(f"Schema:")
df.printSchema()

print(f"\nSample rows:")
df.show(5)

## 4. Feature Engineering

All features are computed **per ticker** using PySpark window functions,
partitioned by ticker and ordered by date.

| Feature | Description |
|---------|-------------|
| `return_1/2/3/5` | Lagged direct returns from close price |
| `rolling_mean_5` | 5-period rolling mean of close |
| `rolling_mean_10` | 10-period rolling mean of close |
| `rolling_std_5` | 5-period rolling standard deviation |
| `momentum_5` | 5-period price momentum (close - close[t-5]) |
| `price_to_mean_5` | Ratio of current close to 5-period mean |


In [ ]:
# Define partitioned window per ticker, ordered by date
window = Window.partitionBy("ticker").orderBy("date")

# --- Lagged returns ---
lags = [1, 2, 3, 5]

df = df.select(
    "*",
    *[
        (
            (lag("close", i).over(window) - lag("close", i + 1).over(window)) /
            lag("close", i + 1).over(window)
        ).alias(f"return_{i}")
        for i in lags
    ]
)

In [ ]:
# --- Rolling means (5-day and 10-day) ---
window_prev_5 = Window.partitionBy("ticker").orderBy("date").rowsBetween(-5, -1)
window_prev_10 = Window.partitionBy("ticker").orderBy("date").rowsBetween(-10, -1)

df = df.select(
    "*",
    avg("close").over(window_prev_5).alias("rolling_mean_5"),
    avg("close").over(window_prev_10).alias("rolling_mean_10"),
)

In [ ]:
# --- Rolling standard deviation (5-day volatility proxy) ---
df = df.select(
    "*",
    std("close").over(window_prev_5).alias("rolling_std_5"),
)

In [ ]:
# --- Momentum: difference between current and 5-day lagged close ---
df = df.select(
    "*",
    (col("close") - lag("close", 5).over(window)).alias("momentum_5")
)

In [ ]:
# --- Price-to-mean ratio: signals deviation from recent trend ---
df = df.withColumn(
    "price_to_mean_5",
    col("close") / col("rolling_mean_5")
)

## 5. Data Preparation & Target Construction

Select model features, drop rows with NaN values (caused by window look-back),
and construct the binary target variable.

**Target definition:**  
`target = 1` if `close[t+3] > close[t]`, else `0`  
Using a 3-period forward horizon to reduce noise and capture short-term trends.


In [ ]:
# Select only the columns needed for the model
FEATURE_COLS = [
    "date", "ticker", "close",
    "rolling_mean_5", "rolling_mean_10",
    "rolling_std_5", "momentum_5", "price_to_mean_5"
]

df_model = df.select(FEATURE_COLS)

# Drop rows where any feature is null (look-back window warm-up period)
df_model = df_model.dropna(subset=[
    "rolling_mean_5", "rolling_mean_10",
    "rolling_std_5", "momentum_5", "price_to_mean_5"
])

In [ ]:
# Construct 3-period forward close price
df_model = df_model.withColumn(
    "future_close",
    lead("close", 3).over(window)
)

# Binary target: 1 if price goes up, 0 otherwise
df_model = df_model.withColumn(
    "target",
    when(col("future_close") > col("close"), 1).otherwise(0)
)

# Drop the auxiliary future_close column
df_model = df_model.drop("future_close")

# Check target distribution
print("Target distribution:")
df_model.groupBy("target").count().show()

## 6. Temporal Train / Test Split

Using a **temporal cutoff split** (not random) to respect the time-series nature
of the data and avoid lookahead bias.

- **Train:** data on or before 2019-01-01
- **Test:** data after 2019-01-01


In [ ]:
CUTOFF_DATE = "2019-01-01"

df_train = df_model.filter(col("date") <= CUTOFF_DATE)
df_test  = df_model.filter(col("date") >  CUTOFF_DATE)

print(f"Train rows: {df_train.count():,}")
print(f"Test rows:  {df_test.count():,}")

In [ ]:
# Optimise Spark for Parquet export
spark.conf.set("spark.sql.parquet.mergeSchema", "false")
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

# Export labelled dataset to S3 silver layer (train + test combined with split column)
SILVER_PATH = f"{BUCKET}silver/etf_ml_features/v1/"

df_train_export = df_train.withColumn("split", lit("train"))
df_test_export  = df_test.withColumn("split",  lit("test"))

df_export = df_train_export.unionByName(df_test_export)
df_export.coalesce(1).write.mode("overwrite").parquet(SILVER_PATH)

print(f"Feature dataset exported to: {SILVER_PATH}")

## 7. Load Feature Dataset for scikit-learn Training

Read back the silver layer into Pandas for scikit-learn model training.


In [ ]:
# Read the silver layer back as Pandas for sklearn
df_all_pd = pd.read_parquet(SILVER_PATH, engine="pyarrow")

# Split by the 'split' column written in the previous step
if "split" in df_all_pd.columns:
    df_train_pd = df_all_pd[df_all_pd["split"] == "train"].copy()
    df_test_pd  = df_all_pd[df_all_pd["split"] == "test"].copy()
else:
    # Fallback: random 80/20 split
    df_train_pd = df_all_pd.sample(frac=0.8, random_state=42)
    df_test_pd  = df_all_pd.drop(df_train_pd.index)

# Define feature columns (exclude target and metadata)
FEATURE_COLS_SKLEARN = [
    "rolling_mean_5", "rolling_mean_10",
    "rolling_std_5", "momentum_5", "price_to_mean_5"
]

# Build X / y for train and test
X_train_pd = df_train_pd[FEATURE_COLS_SKLEARN].copy()
y_train_pd = df_train_pd[["target"]].copy()

X_test_pd  = df_test_pd[FEATURE_COLS_SKLEARN].copy()
y_test_pd  = df_test_pd[["target"]].copy()

y_train_vec = y_train_pd.values.ravel()
y_test_vec  = y_test_pd.values.ravel()

print(f"X_train: {X_train_pd.shape} | X_test: {X_test_pd.shape}")

## 8. Model Training & Evaluation

Three classification models are trained and compared.  
Each is evaluated on the held-out test set using Recall, Precision, and F1.

> **Note on metric choice:** Recall is prioritised over Precision in this context —
> missing a true price increase (false negative) is considered more costly
> than a false alarm (false positive) for the use case explored here.


In [ ]:
results = []  # Will collect metrics for all models

### 8.1 Logistic Regression

In [ ]:
# Logistic Regression with standard scaling
logreg_model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=500, random_state=42))
])

logreg_model.fit(X_train_pd, y_train_vec)
logreg_preds = logreg_model.predict(X_test_pd)

results.append({
    "Model":     "Logistic Regression",
    "Accuracy":  accuracy_score(y_test_vec, logreg_preds),
    "Precision": precision_score(y_test_vec, logreg_preds, zero_division=0),
    "Recall":    recall_score(y_test_vec, logreg_preds, zero_division=0),
    "F1":        f1_score(y_test_vec, logreg_preds, zero_division=0)
})

print("=== Logistic Regression ===")
print(classification_report(y_test_vec, logreg_preds, zero_division=0))

### 8.2 Decision Tree

In [ ]:
# Decision Tree with depth and leaf constraints to avoid overfitting
tree_model = DecisionTreeClassifier(
    max_depth=6,
    min_samples_leaf=50,
    random_state=42
)

tree_model.fit(X_train_pd, y_train_vec)
tree_preds = tree_model.predict(X_test_pd)

results.append({
    "Model":     "Decision Tree",
    "Accuracy":  accuracy_score(y_test_vec, tree_preds),
    "Precision": precision_score(y_test_vec, tree_preds, zero_division=0),
    "Recall":    recall_score(y_test_vec, tree_preds, zero_division=0),
    "F1":        f1_score(y_test_vec, tree_preds, zero_division=0)
})

print("=== Decision Tree ===")
print(classification_report(y_test_vec, tree_preds, zero_division=0))

### 8.3 XGBoost

In [ ]:
# XGBoost gradient boosted classifier
bst = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    objective="binary:logistic",
    random_state=42
)

bst.fit(X_train_pd, y_train_pd)
preds = bst.predict(X_test_pd)

results.append({
    "Model":     "XGBoost",
    "Accuracy":  accuracy_score(y_test_vec, preds),
    "Precision": precision_score(y_test_vec, preds, zero_division=0),
    "Recall":    recall_score(y_test_vec, preds, zero_division=0),
    "F1":        f1_score(y_test_vec, preds, zero_division=0)
})

print("=== XGBoost ===")
print(classification_report(y_test_vec, preds, zero_division=0))

## 9. Model Comparison

In [ ]:
results_df = pd.DataFrame(results).sort_values(by="F1", ascending=False).reset_index(drop=True)

print("=== Model Comparison (sorted by F1) ===")
display(results_df)

best_model = results_df.iloc[0]["Model"]
print(f"\nBest model by F1: {best_model}")

## 10. Export Results to S3 (Gold Layer)

Exports three datasets to the gold layer for Power BI consumption:
1. **Predictions sample** — 200k test rows with predictions from all 3 models
2. **Model metrics** — summary table with Accuracy, Precision, Recall, F1
3. **Classification report** — per-class breakdown for each model


In [ ]:
GOLD_BASE = f"{BUCKET}gold/model_evaluation/v1/"
S3_BUCKET_NAME = os.environ["STRATA_S3_BUCKET"]  # loaded from .env
s3 = boto3.client("s3")
models_dict = {
    "Logistic Regression": logreg_preds,
    "Decision Tree":       tree_preds,
    "XGBoost":             preds
}

In [ ]:
# --- 10.1 Prediction sample (200k rows) ---

LOCAL_PREDS = "/home/sagemaker-user/model_predictions_sample.parquet"
S3_PREDS_KEY = "gold/model_evaluation/v1/model_predictions/model_predictions_sample.parquet"

# Sample from the Spark test dataframe before converting to Pandas
df_test_sample = (
    df_test
    .select(["date", "ticker", "target"] + FEATURE_COLS_SKLEARN)
    .orderBy(rand(seed=42))
    .limit(200_000)
)
test_sample_pd = df_test_sample.toPandas().reset_index(drop=True).copy()

# Ensure clean types
test_sample_pd["date"]   = test_sample_pd["date"].astype(str)
test_sample_pd["ticker"] = test_sample_pd["ticker"].astype(str)
test_sample_pd["target"] = test_sample_pd["target"].astype(int)
for c in FEATURE_COLS_SKLEARN:
    test_sample_pd[c] = pd.to_numeric(test_sample_pd[c], errors="coerce")
test_sample_pd = test_sample_pd.dropna(subset=FEATURE_COLS_SKLEARN + ["target"])

X_sample = test_sample_pd[FEATURE_COLS_SKLEARN].copy()

# Generate predictions from all models on the sample
predictions_df = pd.DataFrame({
    "date":         test_sample_pd["date"].values,
    "ticker":       test_sample_pd["ticker"].values,
    "actual_target":test_sample_pd["target"].values.astype(int),
    "pred_logreg":  logreg_model.predict(X_sample).astype(int),
    "pred_tree":    tree_model.predict(X_sample).astype(int),
    "pred_xgb":     bst.predict(X_sample).astype(int),
})

# Add correct/incorrect flag columns for Power BI
for model_col, pred_col in [("correct_logreg","pred_logreg"),("correct_tree","pred_tree"),("correct_xgb","pred_xgb")]:
    predictions_df[model_col] = (predictions_df["actual_target"] == predictions_df[pred_col]).astype(int)

predictions_df.to_parquet(LOCAL_PREDS, index=False, engine="pyarrow", compression="snappy")
s3.upload_file(LOCAL_PREDS, S3_BUCKET_NAME, S3_PREDS_KEY)
print(f"Predictions exported: {len(predictions_df):,} rows → s3://{S3_BUCKET_NAME}/{S3_PREDS_KEY}")

In [ ]:
# --- 10.2 Model metrics summary ---

LOCAL_METRICS = "/home/sagemaker-user/model_metrics.parquet"
S3_METRICS_KEY = "gold/model_evaluation/v1/model_metrics/model_metrics.parquet"

metrics_rows = []
for model_name, y_pred in models_dict.items():
    metrics_rows.append({
        "model":     model_name,
        "accuracy":  accuracy_score(y_test_vec, y_pred),
        "precision": precision_score(y_test_vec, y_pred, zero_division=0),
        "recall":    recall_score(y_test_vec, y_pred, zero_division=0),
        "f1":        f1_score(y_test_vec, y_pred, zero_division=0)
    })

model_metrics_df = pd.DataFrame(metrics_rows).sort_values("f1", ascending=False).reset_index(drop=True)
model_metrics_df.to_parquet(LOCAL_METRICS, index=False, engine="pyarrow", compression="snappy")
s3.upload_file(LOCAL_METRICS, S3_BUCKET_NAME, S3_METRICS_KEY)
display(model_metrics_df)
print(f"Metrics exported → s3://{S3_BUCKET_NAME}/{S3_METRICS_KEY}")

In [ ]:
# --- 10.3 Full classification report per model ---

LOCAL_REPORT = "/home/sagemaker-user/classification_report.parquet"
S3_REPORT_KEY = "gold/model_evaluation/v1/classification_report/classification_report.parquet"

report_rows = []
for model_name, y_pred in models_dict.items():
    report = classification_report(y_test_vec, y_pred, output_dict=True, zero_division=0)
    for class_label in ["0", "1", "macro avg", "weighted avg"]:
        report_rows.append({
            "model":     model_name,
            "class":     class_label,
            "precision": report[class_label]["precision"],
            "recall":    report[class_label]["recall"],
            "f1_score":  report[class_label]["f1-score"],
            "support":   report[class_label]["support"]
        })

classification_report_df = pd.DataFrame(report_rows).reset_index(drop=True)
classification_report_df.to_parquet(LOCAL_REPORT, index=False, engine="pyarrow", compression="snappy")
s3.upload_file(LOCAL_REPORT, S3_BUCKET_NAME, S3_REPORT_KEY)
display(classification_report_df)
print(f"Classification report exported → s3://{S3_BUCKET_NAME}/{S3_REPORT_KEY}")

## 11. Save Trained Models

Serialise all trained models locally and upload to S3.


In [ ]:
# Save models as joblib files
joblib.dump(logreg_model,  "logreg_model.joblib")
joblib.dump(tree_model,    "tree_model.joblib")
joblib.dump(bst,           "xgboost_model.joblib")

# Upload to S3
for filename in ["logreg_model.joblib", "tree_model.joblib", "xgboost_model.joblib"]:
    s3.upload_file(filename, S3_BUCKET_NAME, f"gold/models/{filename}")
    print(f"Uploaded: gold/models/{filename}")

---

## Results Summary

| Model | Recall | Precision | F1 |
|-------|--------|-----------|-----|
| Logistic Regression | 0.975 | 0.551 | 0.704 |
| Decision Tree | 0.855 | 0.558 | 0.675 |
| XGBoost | 0.838 | 0.559 | 0.671 |

**Best model:** Logistic Regression (F1: 0.70, Recall: 0.98)

The high recall / lower precision trade-off is expected in financial classification tasks
with noisy market data and a relatively simple feature set.
Future work could explore additional features (volume, sector, macro indicators)
and more sophisticated time-series models.

---
*Authors: Tomás Morales Galván · Miguel Bachiller Segovia*
